# Set up

## Import packages

In [ ]:
import pandas as pd
import numpy as np

## Locate lakehouse
Using the `notebookutils` package we can locate the lakehouse located in our specified storage workspace using its `workspaceId`. THe use of a fixed workspace containing the lakehouse that is used by both the production and the development environment was made to simplify deployment as currently the fabric CI/CD tools are quite limited and require further cost and subscription to be able to fully integrate such a solution where everything is parameterized.

In [ ]:
storage_workspace_id = "29cfe25d-afd5-473a-91d8-1a13b9efdddf"
lakehouse_artifact = notebookutils.lakehouse.get(name="DataLakehouse", workspaceId=storage_workspace_id)

lakehouse_id = lakehouse_artifact.id

## Load data

In the same way we did during the data preprocessing we start by importing the data from the lakehouse using spark and converting it to a pandas dataframe.

In [ ]:
lakehouse_schema = "PreProcessedData"
path_to_schema = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/"

# PIS data
aggregated_customers_df = spark.read.format("delta").load(path_to_schema + "Aggregated_CustomerData").toPandas()

# DTU data
courses_df = spark.read.format("delta").load(path_to_schema + "Preproccesed_CourseData").toPandas()
students_df = spark.read.format("delta").load(path_to_schema + "Preproccesed_StudentData").toPandas()
#student_admission_df = spark.read.format("delta").load(path_to_schema + "").toPandas()
#study_program_df = spark.read.format("delta").load(path_to_schema + "").toPandas()

## Process Data 

### Calenda filling
In order to achive visuals that show all months in the later visualization in PowerBI we add 2 entries into all months that are missing an entry within the date rage of the agregated data.

This is done by creating 2 date range for that interval one where an entry is placed at the end and one where is placed at the begining, this allows for the visual to continuosly exists no matter where the date is sliced along the month during visualization.

In [ ]:
min_date = aggregated_customers_df['time_stamp'].min()
max_date = aggregated_customers_df['time_stamp'].max()

starts = pd.date_range(start=min_date, end=max_date, freq='MS')
ends = pd.date_range(start=min_date, end=max_date, freq='M')
full_months = pd.DataFrame({'time_stamp': starts.union(ends).sort_values()})

This calendar of missing months stars and ends is then filtered against a series of the existing months yielding only the missing months as a result.

In [ ]:
existing_months = aggregated_customers_df['time_stamp'].dt.to_period('M')
full_months_filtered = full_months[~full_months['time_stamp'].dt.to_period('M').isin(existing_months)].copy()
full_months_filtered['time_stamp'] = full_months_filtered['time_stamp'].astype('datetime64[us]')

Finally the missing months dataframe is concatenated into the agregated customer data dataframe, sorted and reindexed.

In [ ]:
aggregated_customers_df = pd.concat([aggregated_customers_df, full_months_filtered], ignore_index=True)
aggregated_customers_df = aggregated_customers_df.sort_values('time_stamp').reset_index(drop=True)

### Cross referencing
During this step the customer date and the DTU sourced data are merged based on the `course_number` and `s_number` to gain the `gender, direction_EN, direction_DK, education_EN, education_DK, course_name`.

In [ ]:
cross_referenced_df = aggregated_customers_df.merge(courses_df[['course_number', 'course_name']], on='course_number', how='left', )
cross_referenced_df = cross_referenced_df.merge(students_df[['s_number', 'gender','direction_EN','direction_DK','education_EN','education_DK']], on='s_number', how='left')

### New column engineering
In this step many columns are created which will then be useful for visualization.

Firstly specific date column are created, this will aid visualization allowing for the creation of a date hierarchy during the visualization steps. The following date columns are crearted:
- `time_slot`
- `weekday`
- `weekday_n`
- `month`
- `month_n`
- `semester`
- `semester_n`
- `year`

The `_n` variants are created to allow to set the ordering during visualization to be based on a integer value defined by them insted of alphabeticaly base on the name of the date variable.

In [ ]:
cross_referenced_df['weekday'] = cross_referenced_df['time_stamp'].dt.day_name()
cross_referenced_df['weekday_n'] = cross_referenced_df['time_stamp'].dt.day_of_week
cross_referenced_df['month'] = cross_referenced_df['time_stamp'].dt.month_name()
cross_referenced_df['month_n'] = cross_referenced_df['time_stamp'].dt.month
cross_referenced_df['year'] = cross_referenced_df['time_stamp'].dt.year


cross_referenced_df['semester'] = np.where(
    cross_referenced_df['time_stamp'].dt.month <= 8,
    'Spring ' + cross_referenced_df['time_stamp'].dt.year.astype(str).str[2:],
    'Fall ' + cross_referenced_df['time_stamp'].dt.year.astype(str).str[2:]
)
cross_referenced_df['semester_n'] = np.where(
    cross_referenced_df['time_stamp'].dt.month <= 8,
    cross_referenced_df['time_stamp'].dt.year.astype(str).str[2:] + '_0',
    cross_referenced_df['time_stamp'].dt.year.astype(str).str[2:] + '_1 '
)

cross_referenced_df['time_slot'] = np.where(
    cross_referenced_df['time_stamp'].dt.time < pd.Timestamp('12:30:00').time(),
    'Morning',
    'Afternoon'
)

Next, an estimation of the study start is made base on the first 2 numbers of the `s_number` as this denotes the year it was created.

In [ ]:
study_mid = cross_referenced_df['s_number'].astype(str).str[1:3]
verified = cross_referenced_df['s_number'].isin(students_df['s_number'])

cross_referenced_df['study_start'] = np.where(
    (pd.to_numeric(study_mid, errors='coerce') >= 23) & verified,
    's' + study_mid,
    'N/A'
)

### Data normalization
Lastly, normalize missing data by filling NA/Blank with `Other`.

In [ ]:
cross_referenced_df[['gender']] = cross_referenced_df[['gender']].fillna('Other')
cross_referenced_df[['direction_EN','education_EN', 'direction_DK','education_DK']] = cross_referenced_df[['direction_EN','education_EN', 'direction_DK','education_DK']].fillna('N/A')
cross_referenced_df[['course_number', 'course_name']] = cross_referenced_df[['course_number', 'course_name']].fillna('N/A')
cross_referenced_df[['workshop_attendance']] = cross_referenced_df[['workshop_attendance']].fillna(False)

In [ ]:
cross_referenced_df

### Save data to Lakehouse

In [ ]:
# Set up lakehouse destination parameters. Create schema if missing
lakehouse_schema = 'ProcessedData'
path_to_schema = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}"

if notebookutils.fs.exists(path_to_schema):
    notebookutils.fs.mkdirs(path_to_schema)

In [ ]:
cross_referenced_data = spark.createDataFrame(cross_referenced_df)

In [ ]:
table_name = 'Processed_CustomerData'
abfss_path = f"abfss://{storage_workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{lakehouse_schema}/{table_name}"

cross_referenced_data.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(abfss_path)